# SQL Analysis

This notebook executes SQL queries against the cleaned supermarket datasets using DuckDB.

The objective is to generate business insights and create result tables that can later be used in Power BI and the project report.


In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
query = """
SELECT *
FROM read_csv_auto('../data/cleaned/aldi_clean.csv')
LIMIT 5;
"""

result = con.execute(query).fetchdf()
result

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
0,Aldi,3.09,0.14,unit,Mamia Ultra-fit Peppa Pig Nappy Pants 22 Pack/...,20240413,baby_products,False
1,Aldi,3.09,0.17,unit,Mamia Ultra-fit Peppa Pig Nappy Pants 18 Pack/...,20240413,baby_products,False
2,Aldi,3.59,0.09,unit,Mamia Ultra-fit Nappy Pants 40 Pack/Size 4,20240413,baby_products,False
3,Aldi,4.79,0.32,unit,Mamia Boy's Night Pants 15 Pack,20240413,baby_products,False
4,Aldi,4.79,0.32,unit,Mamia Girl's Night Pants 15 Pack,20240413,baby_products,False


In [4]:
query = """
SELECT ROUND(AVG("prices_(£)"), 2) AS avg_price
FROM read_csv_auto('../data/cleaned/aldi_clean.csv');
"""

con.execute(query).fetchdf()

,avg_price
0,2.22


In [5]:
query = """
SELECT supermarket,
       ROUND(AVG(price), 2) AS avg_price
FROM (
    SELECT 'Aldi' AS supermarket, "prices_(£)" AS price
    FROM read_csv_auto('../data/cleaned/aldi_clean.csv')

    UNION ALL

    SELECT 'ASDA', "prices_(£)"
    FROM read_csv_auto('../data/cleaned/asda_clean.csv')

    UNION ALL

    SELECT 'Morrisons', "prices_(£)"
    FROM read_csv_auto('../data/cleaned/morrisons_clean.csv')

    UNION ALL

    SELECT 'Sainsbury''s', "prices_(£)"
    FROM read_csv_auto('../data/cleaned/sains_clean.csv')

    UNION ALL

    SELECT 'Tesco', "prices_(£)"
    FROM read_csv_auto('../data/cleaned/tesco_clean.csv')
) AS all_products
GROUP BY supermarket
ORDER BY avg_price ASC;
"""

avg_prices = con.execute(query).fetchdf()
avg_prices

,supermarket,avg_price
0,Aldi,2.22
1,Morrisons,4.89
2,Tesco,5.28
3,Sainsbury's,5.46
4,ASDA,5.76


## Average Price by Supermarket

The SQL analysis showed substantial differences in average product prices across the five supermarkets.

### Key findings

* **Aldi** had the lowest average product price (£2.22).
* **ASDA** had the highest average product price (£5.76).
* **Sainsbury's** (£5.46) and **Tesco** (£5.28) were positioned in the higher-price segment.
* **Morrisons** (£4.89) sat between Aldi and the other major supermarkets.

### Business interpretation

The results suggest that Aldi operates with a significantly lower-priced product mix, while ASDA and Sainsbury's appear to offer a broader range of higher-priced products.

These findings are based on average product prices across all available categories and do not account for differences in product assortment or package sizes.


In [6]:
query = """
SELECT category,
       ROUND(AVG(price), 2) AS avg_category_price
FROM (
    SELECT category, "prices_(£)" AS price
    FROM read_csv_auto('../data/cleaned/aldi_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/asda_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/morrisons_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/sains_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/tesco_clean.csv')
) AS all_products
GROUP BY category
ORDER BY avg_category_price DESC;
"""

category_prices = con.execute(query).fetchdf()
category_prices

,category,avg_category_price
0,home,10.00
1,drinks,7.49
2,health_products,7.14
3,household,5.36
4,baby_products,5.09
5,pets,4.85
6,free-from,3.25
7,frozen,2.87
8,fresh_food,2.80
9,bakery,2.63


## Average Price by Product Category

The SQL analysis identified significant differences in average product prices across product categories.

### Key findings

- **Home** had the highest average price at £10.00.
- **Drinks** (£7.49) and **Health Products** (£7.14) were the next highest-priced categories.
- **Food Cupboard** had the lowest average price at £2.16.
- **Bakery** (£2.63), **Fresh Food** (£2.80), and **Frozen** (£2.87) also had relatively low average prices.

### Business interpretation

The results suggest that average product prices vary considerably by category. Non-food categories such as Home, Health Products and Household generally have higher average prices, while everyday food categories such as Food Cupboard, Bakery and Fresh Food have lower average prices.

However, average price alone does not account for differences in product type, package size, quantity or product assortment. Therefore, these results should be interpreted as descriptive rather than a direct comparison of value for money between categories.

In [7]:
query = """
SELECT supermarket,
       COUNT(*) AS product_count
FROM (
    SELECT 'Aldi' AS supermarket, *
    FROM read_csv_auto('../data/cleaned/aldi_clean.csv')

    UNION ALL

    SELECT 'ASDA', *
    FROM read_csv_auto('../data/cleaned/asda_clean.csv')

    UNION ALL

    SELECT 'Morrisons', *
    FROM read_csv_auto('../data/cleaned/morrisons_clean.csv')

    UNION ALL

    SELECT 'Sainsbury''s', *
    FROM read_csv_auto('../data/cleaned/sains_clean.csv')

    UNION ALL

    SELECT 'Tesco', *
    FROM read_csv_auto('../data/cleaned/tesco_clean.csv')
) AS all_products
GROUP BY supermarket
ORDER BY product_count DESC;
"""

product_counts = con.execute(query).fetchdf()
product_counts

,supermarket,product_count
0,Sainsbury's,2600289
1,ASDA,2456407
2,Tesco,2189783
3,Morrisons,1794065
4,Aldi,464863


## Product Record Count by Supermarket

The SQL analysis compared the number of cleaned product records available for each supermarket.

### Key findings

* **Sainsbury's** contained the largest dataset with **2,600,289 records**.
* **ASDA** was the second largest with **2,456,407 records**.
* **Tesco** contained **2,189,783 records** after duplicate removal.
* **Morrisons** contributed **1,794,065 records**.
* **Aldi** had a substantially smaller dataset with **464,863 records**.

### Business interpretation

The large differences in record counts indicate that the supermarkets do not contribute equal amounts of data. Sainsbury's and ASDA have much broader product coverage in this dataset, while Aldi has a considerably smaller product catalogue.

These differences are important when interpreting average prices and category comparisons because supermarkets with larger product ranges may include more premium and specialist products.


In [3]:
query = """
SELECT category,
       ROUND(AVG(price), 2) AS avg_category_price
FROM (
    SELECT category, "prices_(£)" AS price
    FROM read_csv_auto('../data/cleaned/aldi_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/asda_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/morrisons_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/sains_clean.csv')

    UNION ALL

    SELECT category, "prices_(£)"
    FROM read_csv_auto('../data/cleaned/tesco_clean.csv')
) AS all_products
GROUP BY category
ORDER BY avg_category_price DESC;
"""

category_prices = con.execute(query).fetchdf()
category_prices

,category,avg_category_price
0,home,10.00
1,drinks,7.49
2,health_products,7.14
3,household,5.36
4,baby_products,5.09
5,pets,4.85
6,free-from,3.25
7,frozen,2.87
8,fresh_food,2.80
9,bakery,2.63


**Data quality note:** The `home` category is not present in all supermarkets, so comparisons involving this category are based on a smaller subset of the data.


## Average Price by Category

The SQL analysis compared average product prices across all product categories.

### Key findings

* The **home** category had the highest average price (£10.00).
* **Drinks** (£7.49) and **health_products** (£7.14) were also among the most expensive categories.
* **Food_cupboard** had the lowest average price (£2.16).
* **Bakery**, **fresh_food**, and **frozen** products were clustered in the lower-price range (£2.63–£2.87).

### Business interpretation

The results suggest that discretionary and specialist categories such as home products, alcoholic beverages, and health-related products carry significantly higher average prices than everyday grocery staples.

The exceptionally high value for the **home** category should be interpreted with caution because this category appears only in some supermarkets (ASDA, Morrisons, and Sainsbury's) and contains a smaller number of records, which can increase the average price.

Overall, the analysis indicates that core grocery categories remain relatively low priced, while lifestyle and specialist categories contribute disproportionately to total spending.


In [4]:
query = """
SELECT supermarket,
       COUNT(*) AS total_products,
       SUM(CASE WHEN own_brand = TRUE THEN 1 ELSE 0 END) AS own_brand_products,
       ROUND(
           100.0 * SUM(CASE WHEN own_brand = TRUE THEN 1 ELSE 0 END) / COUNT(*),
           2
       ) AS own_brand_percentage
FROM (
    SELECT supermarket, own_brand
    FROM read_csv_auto('../data/cleaned/aldi_clean.csv')

    UNION ALL

    SELECT supermarket, own_brand
    FROM read_csv_auto('../data/cleaned/asda_clean.csv')

    UNION ALL

    SELECT supermarket, own_brand
    FROM read_csv_auto('../data/cleaned/morrisons_clean.csv')

    UNION ALL

    SELECT supermarket, own_brand
    FROM read_csv_auto('../data/cleaned/sains_clean.csv')

    UNION ALL

    SELECT supermarket, own_brand
    FROM read_csv_auto('../data/cleaned/tesco_clean.csv')
) AS all_products
GROUP BY supermarket
ORDER BY own_brand_percentage DESC;
"""

own_brand = con.execute(query).fetchdf()
own_brand

,supermarket,total_products,own_brand_products,own_brand_percentage
0,ASDA,2456407,714547.0,29.09
1,Tesco,2189783,567743.0,25.93
2,Morrisons,1794065,436743.0,24.34
3,Sains,2600289,593802.0,22.84
4,Aldi,464863,79140.0,17.02


## Own-Brand Share by Supermarket

The SQL analysis measured the proportion of products classified as own-brand within each supermarket dataset.

### Key findings

* **ASDA** had the highest own-brand share (**29.09%**).
* **Tesco** ranked second (**25.93%**).
* **Morrisons** followed with **24.34%**.
* **Sainsbury's** recorded **22.84%**.
* **Aldi** had the lowest own-brand share (**17.02%**).

### Business interpretation

The results suggest that ASDA relies most heavily on own-brand products within this dataset, while Aldi shows the lowest recorded proportion of own-brand items.

This finding should be interpreted cautiously. Aldi is widely known for a strong private-label strategy, so the lower percentage may reflect differences in how products were classified in the source data rather than Aldi's true commercial positioning.

The analysis demonstrates the importance of validating analytical results against domain knowledge and considering potential data classification issues.
